In [1]:
import torch
import numpy as np
import os
import shutil
import pandas as pd
from torchvision import models
import gc

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
torch.cuda.empty_cache()

cuda


In [2]:
import torch
import sys
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

/home/undergrad/2026/ojonesma/Spring2026/EyeClassifierEnv/bin/python
2.10.0+cu126
12.6
True


## Efficient Net
#### Olivia Jones-Martin

In [3]:
from utilities import *

# setting path variables for later use
TRAINING_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/a. Training Set"
VALIDATION_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/b. Validation Set"
TESTING_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/c. Testing Set"

GROUND_TRUTHS_PATH = "./A. RFMiD_All_Classes_Dataset/3. GroundtruthsChallenge"
'''
GROUND_TRUTHS_PATH can also point to the folder containing ground truths for all classes
The way my dataset is set up is that there's 3 folders, one for the images, one for the metadata for all classes,
and one for the metadata in which any disease with less than 10 instances get grouped into the label other
This can be modified depending on which set of metadata you're using
-Olivia Jones-Martin
'''

TRAINING_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "a. RFMiD_Training_Labels.csv")
VALIDATION_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "b. RFMiD_Validation_Labels.csv")
TEST_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "c. RFMiD_Testing_Labels.csv")

Loss function being used is [binary cross entropy with logits loss](https://docs.pytorch.org/docs/1.7.1/generated/torch.nn.BCEWithLogitsLoss.html#torch.nn.BCEWithLogitsLoss). Weights are calculated for the pos_weight parameter for the function.

In [4]:
# Get counts for each of the

import math

trainingMetadata = pd.read_csv(TRAINING_METADATA_PATH)
diseaseLabels = trainingMetadata.columns[1:]

# counts includes every class except for ID
counts = trainingMetadata.iloc[:, 1:].sum()
print("Disease Counts:")
print(counts)

# total amount of images and labels in the training set
image_count = trainingMetadata.shape[0]
label_count = counts.shape[0]
print(f"Image count: {image_count}")
print(f"Label count: {label_count}")

'''
If there is an instance where a disease appears 0 times, give a warning and
set the count value to one to prevent a divide by zero error when calculating
the weights
'''
for i in range(label_count):
    if counts.iloc[i] == 0:
        counts.iloc[i] = 1
        print(f"WARNING! Zero instances of {diseaseLabels[i]} in dataset")

'''
Calculate pos_weights for binary cross entropy with logits loss
pos weight for a class should be (negative counts of class)/(positive counts of class)
Documentation in reference 2
'''
pos_weights = [math.log1p((image_count - count)/count) for count in counts]
print("pos_weigths for BCEWithLogitsLoss")
print(pos_weights)

Disease Counts:
Disease_Risk    1519
DR               376
ARMD             100
MH               317
DN               138
MYA              101
BRVO              73
TSLN             186
ERM               14
LS                47
MS                15
CSR               37
ODC              282
CRVO              28
TV                 6
AH                16
ODP               65
ODE               58
ST                 5
AION              17
PT                11
RT                14
RS                43
CRS               32
EDN               15
RPEC              22
MHL               11
RP                 6
OTHER             34
dtype: int64
Image count: 1920
Label count: 29
pos_weigths for BCEWithLogitsLoss
[0.2342729624260544, 1.6304913216319328, 2.9549102790337356, 1.8011786911445467, 2.6328267798646223, 2.9449599481805677, 3.269621023873436, 2.3343337913086257, 4.9210231354065685, 3.7099328633117685, 4.852030263919617, 3.9491625523776026, 1.9181733940837136, 4.227875954846623, 5.76832099579377

Load the images

In [ ]:
from torch.utils.data import DataLoader, Subset

# metadata for training dataset already loaded in previous parts, only need to load in validatoin and
# test metadata dataframes
validationMetadata = pd.read_csv(VALIDATION_METADATA_PATH)
testMetadata = pd.read_csv(TEST_METADATA_PATH)

# Number of modules used for bagging (bootstrap aggregating)
NUM_OF_MODULES = 5
BATCH_SIZE = 20
NUM_OF_WORKERS = 0
EPOCH_COUNT = 15

training_set_size = trainingMetadata.shape[0]
validation_set_size = validationMetadata.shape[0]
test_set_size = testMetadata.shape[0]

'''
efficientnet_b1 uses images sizes of 224x224 (reference 3),
normilization used is RGB means of [0.485, 0.456, 0.406] and
standard deviations of [0.229, 0.224, 0.225]. means and std
deviation are given by reference 1 in section 5.2
'''
augmenter = dataAugmenter(image_size = (224, 224),
                          norm_mean=(0.485, 0.456, 0.406),
                          norm_std=(0.229, 0.224, 0.225),
                          useCutOut=True)
# Augmenter class used to provide transformations

training_dataset = FundusImageDataset(metadata=trainingMetadata,
                                      image_directory=TRAINING_SET_PATH,
                                      transform=augmenter.transform_train)
validation_dataset = FundusImageDataset(metadata=validationMetadata,
                                        image_directory=VALIDATION_SET_PATH,
                                        transform=augmenter.transform_test)
test_dataset = FundusImageDataset(metadata=testMetadata,
                                  image_directory=TESTING_SET_PATH,
                                  transform=augmenter.transform_test)

'''
Dataloader takes a "Map-style dataset" that has the __getitem__() and __len__() protocols
to access data
this class is implemented in utilities.py
multiple random subsets of training and validation sets are picked out for bootstrap sampling, testing_loader can be created here though
'''
testing_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Model Setup
setting up the use of efficientNet v3, using ensamble/bagging

In [6]:
# this code was generated by copilot to just see how much memory was being used as we were running into issues with the model using too much memory
def memorySummary():
    print("Current GPU Memory Usage:")
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.2f} GB")
    print(f"\nTotal GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [7]:
from torchmetrics.classification import F1Score

def calculate_thresholds(probs, labels, label_count):
    best_thresholds = []

    f1 = F1Score(task='binary')

    for label in range(label_count):
        best_threshold = 0.5
        best_F1 = 0

        label_true = torch.from_numpy(labels[:, label])
        label_probs = torch.from_numpy(probs[:, label])
        for threshold in np.arange(0.05, 0.95, 0.05):
            label_prediction = (label_probs > threshold).float()
            score = f1(label_prediction, label_true)
            if score > best_F1:
                best_F1 = score
                best_threshold = threshold
        best_thresholds.append(best_threshold)

    return best_thresholds

def get_predictions(probs, thresholds):
    '''
    probs is a numpy array and thresholds is a standard python array full of scalars,
    there are as many colums as the size in thresholds
    '''
    probs = torch.from_numpy(probs)
    thresholds = torch.tensor(thresholds).unsqueeze(0)
    return (probs > thresholds).float()

Training the model

In [ ]:
from torchmetrics.classification import MultilabelF1Score

# Create loss function with given weights calculated
pos_weights = torch.FloatTensor(pos_weights).to(device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights).cuda()

# array used to average out predictions
total_probs = np.zeros((test_set_size, label_count))

# targets for using F1 test
targets = torch.from_numpy(testMetadata.iloc[:, 1:].to_numpy().astype(np.float32))

'''
Previous attempt used several modules in a nn.ModuleList datatype but this used too much memory in the server so only one module
is evaluated at a time, cuda cache is also emptied to prevent any issues
'''
gc.collect()
torch.cuda.empty_cache()
for i in range(NUM_OF_MODULES):
    print(f"Module Iteration: {i + 1}")
    model = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)

    # replace feature layer with desired amount of output classes
    in_features = model.classifier[1].in_features
    model.classifier[1] = torch.nn.Linear(in_features, label_count)
    model = model.to(device)

    # set up the optimizer
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    # Create a dataloader for subset of training and validation samples for training, each module uses a different subset
    indices = np.random.choice(a=training_set_size,
                               size=training_set_size,
                               replace=True)

    subset = Subset(training_dataset, indices)
    training_loader = DataLoader(dataset=subset,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_OF_WORKERS,
                                 pin_memory=False)

    validation_loader = DataLoader(dataset=validation_dataset,
                                   batch_size=BATCH_SIZE,
                                   shuffle=True,
                                   num_workers=NUM_OF_WORKERS)

    # Create evaluator for this module
    evaluator = ModelEvaluator(training_loader=training_loader,
                               validation_loader=validation_loader,
                               testing_loader=testing_loader,
                               loss_criterion=criterion,
                               optimizer=optimizer,
                               device=device,
                               lr_scheduler=lr_scheduler)

    # Train model
    print(f"Memory before training module {i+1}:")
    memorySummary()

    evaluator.train(model=model, epoch_count=EPOCH_COUNT, verbose=True)
    
    # Get validation probabilities to compute optimal thresholds
    val_probs, val_labels = evaluator.getValidationProbabilities(model=model)
    thresholds = calculate_thresholds(val_probs, val_labels, label_count)
    
    # Get test probabilities for final evaluation
    probabilities = evaluator.testProbabilities(model=model)
    total_probs += probabilities
    predictions = get_predictions(probabilities, thresholds)

    # get individual F1 score for sub model

    f1 = MultilabelF1Score(num_labels=label_count)
    score = f1(predictions, targets)
    print(f"Model {i+1} F1 score:")
    print(score)

    # get confusion matrices
    matrices = evaluator.test(model)
    print(f"Model {i+1} confusion matrices")
    print(matrices)

    # Clear cuda memory immediately after training
    del model, optimizer, evaluator, training_loader, validation_loader, subset, indices
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Memory after clearing module {i+1}:")
    memorySummary()

average_probs = total_probs / NUM_OF_MODULES
thresholds = calculate_thresholds(average_probs, targets.numpy(), label_count)
print(f"Thresholds: {thresholds}")
print(f"Average probs min/max: {average_probs.min()}, {average_probs.max()}")
predictions = get_predictions(average_probs, thresholds)
print("Average predictions")
print(predictions)

Module Iteration: 1
Memory before training module 1:
Current GPU Memory Usage:
Allocated: 0.03 GB
Reserved: 0.04 GB
Free: 50.86 GB

Total GPU Memory: 50.90 GB
Epoch: 1
Loss: 0.7136559132486582 Accuracy: 0.5561961206896552
Validation Loss: 0.704012768343091 Val Accuracy: 0.6745689655172413
Epoch: 2
Loss: 0.6718387038757404 Accuracy: 0.7639008620689656
Validation Loss: 0.6655097287148237 Val Accuracy: 0.8691810344827586
Model 1 F1 score:
tensor(0.1080)
Model 1 confusion matrices
[[[ 35  99]
  [214 292]]

 [[464  52]
  [105  19]]

 [[577  32]
  [ 29   2]]

 [[412 124]
  [ 59  45]]

 [[512  82]
  [ 39   7]]

 [[558  50]
  [ 29   3]]

 [[593  24]
  [ 19   4]]

 [[481 106]
  [ 40  13]]

 [[534 101]
  [  4   1]]

 [[477 148]
  [  9   6]]

 [[523 110]
  [  7   0]]

 [[579  48]
  [ 11   2]]

 [[482  67]
  [ 71  20]]

 [[602  29]
  [  9   0]]

 [[630   8]
  [  2   0]]

 [[604  31]
  [  5   0]]

 [[375 241]
  [ 16   8]]

 [[596  27]
  [ 16   1]]

 [[574  64]
  [  2   0]]

 [[584  52]
  [  4   0]]

## Accuracy using F1 Test

In [9]:

f1 = MultilabelF1Score(num_labels=label_count)
score = f1(average_predictions, targets)

print(average_predictions)

print("Score")
print(score)

# try oversampling rare class or class weighted sampling
# improve augmentations
# use full validation set instead of sampling subsets
# unfreeze rest of weights for efficientnet
# use recall curve

NameError: name 'average_predictions' is not defined